# E44b — frozen event comparison (owner runs once)

**Do not run this notebook while building the evaluation. Status: plan fixed, not scored.**

1. On the laptop, from the repo, run:
   `rtk proxy env UV_OFFLINE=1 HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 uv run --no-sync python notebooks/e44_score/prepare_bundle.py --output /tmp/e44_kaggle_bundle.zip`.
2. Upload that ZIP to a **private Kaggle Dataset**, attach it here, and keep notebook/output private. The bundle includes frozen annotations and hidden provenance.
3. Use Python >=3.12, CPU and **Internet off**. The frozen ONNX provider is CPU. No training occurs.
4. Exact versions: numpy 2.5.3, pandas 3.0.6, scikit-learn 1.9.1, onnxruntime 1.30.0, torch 2.14.1+cpu, transformers 5.18.0. If not installed, attach their Linux/Python-compatible offline wheels **and dependency wheels**, then set WHEELS below. A mismatch stops; do not relax checks.
5. Run setup and validation first. Run the final prediction/scoring cell **once**, only if E44b has not already been attempted in another session.
6. Save the notebook version and download the complete `/kaggle/working/e44_repo/data/outputs/` folder plus `e44_execution_manifest.json`. Retain partial outputs/locks after a failure; do not clear them or start a fresh session to change a number.

Plan SHA: `8d6f1406db597935ec8387ff7ffb6bc328917bf9031a9497cd8d9e2f74f42193`.
822 AI-agreed rows; 78 disagreements dropped; no independent human validation. Default has 28 rows. A is the existing E24 credit-rule-plus-model reference. Challengers need >=+0.02 primary macro-F1 and paired CI lower >0 vs C; no engine change is made.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import os
import subprocess
import sys
import zipfile

if sys.version_info < (3, 12):
    raise RuntimeError("Use Python >=3.12 and matching offline wheels.")
bundles = list(Path("/kaggle/input").rglob("e44_kaggle_bundle.zip"))
if len(bundles) != 1:
    raise RuntimeError("Attach exactly one private e44_kaggle_bundle.zip dataset.")
BUNDLE = bundles[0]
WHEELS = None  # If needed: Path("/kaggle/input/your-private-wheel-dataset/wheels")
WORK = Path("/kaggle/working/e44_repo")
if WORK.exists():
    raise FileExistsError("Workspace exists. Preserve it; do not restart an attempted run.")
WORK.mkdir()
with zipfile.ZipFile(BUNDLE) as archive:
    for info in archive.infolist():
        destination = (WORK / info.filename).resolve()
        if not destination.is_relative_to(WORK.resolve()):
            raise ValueError("Unsafe archive path")
    archive.extractall(WORK)

def file_sha(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for part in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(part)
    return digest.hexdigest()

bundle_manifest = json.loads((WORK / "e44_bundle_manifest.json").read_text())
for name, expected in bundle_manifest["files_sha256"].items():
    if file_sha(WORK / name) != expected:
        raise ValueError("Upload hash mismatch: " + name)
print("Bundle hashes verified; no prediction or label parsed.")

In [ ]:
requirements = (WORK / "e44_runtime_requirements.txt").read_text().splitlines()
wanted_versions = dict(line.split("==", 1) for line in requirements if line)
def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

mismatches = {name: (installed_version(name), wanted)
              for name, wanted in wanted_versions.items()
              if installed_version(name) != wanted}
if mismatches:
    if WHEELS is None or not WHEELS.is_dir():
        raise RuntimeError("Attach exact-version offline wheels and set WHEELS: " + str(mismatches))
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                    "--find-links", str(WHEELS), "-r",
                    str(WORK / "e44_runtime_requirements.txt")], check=True)
if any(installed_version(name) != wanted for name, wanted in wanted_versions.items()):
    raise RuntimeError("Exact frozen runtime is required.")

os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                  HF_DATASETS_OFFLINE="1", TOKENIZERS_PARALLELISM="false",
                  PYTHONDONTWRITEBYTECODE="1")
os.chdir(WORK)
sys.path.insert(0, str(WORK / "src"))
from riskengine.eval.e44_score import load_plan, verify_inputs

spec = load_plan(WORK)
verify_inputs(WORK, spec, runtime=True)
execution = {
    "bundle_sha256": file_sha(BUNDLE),
    "plan_sha256": bundle_manifest["plan_sha256"],
    "bundled_notebook_sha256": bundle_manifest["files_sha256"][
        "notebooks/e44_score/e44_score.ipynb"],
    "validated_utc": datetime.now(timezone.utc).isoformat(),
    "runtime": wanted_versions,
    "note": "Paths may be edited; bundled notebook hash identifies the supplied source."
}
with Path("/kaggle/working/e44_execution_manifest.json").open("x") as stream:
    json.dump(execution, stream, indent=2)
print("Plan, frozen inputs, weights and runtime verified. No model has run.")

## Execute this cell once

This starts the persistent run lock before loading models. The second command scores the saved predictions with the fixed 1,000 moving-block bootstrap draws. It never changes rules, weights or the engine. Preserve a failed attempt and contact the owner; do not delete locks or rerun.

In [ ]:
environment = dict(os.environ)
environment["PYTHONPATH"] = str(WORK / "src")
for phase in ("--predict", "--score"):
    subprocess.run([sys.executable, "-m", "riskengine.eval.e44_score", phase],
                   cwd=WORK, env=environment, check=True)
result = json.loads((WORK / "data/outputs/e44_results.json").read_text())
print(json.dumps({
    "scores": {candidate: {name: values[name] for name in
               ("macro_f1_primary", "macro_f1_without_other", "accuracy")}
               for candidate, values in result["scores"].items()},
    "decision": result["decision"], "limits": result["limits"],
    "running_system": result["running_system"], "engine_changed": result["engine_changed"]
}, indent=2))
print("Download the output folder and execution manifest; retain every lock.")